In [1]:
!pip -q install fastcoref spacy
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
text = ("Rahul bought a laptop yesterday. He said it was very fast. "
        "Priya liked the laptop too, so she bought it for her sister.")
print(text)

Rahul bought a laptop yesterday. He said it was very fast. Priya liked the laptop too, so she bought it for her sister.


In [3]:
USE_NEURAL = True
try:
    from fastcoref import FCoref
    model = FCoref(device="cpu")
    preds = model.predict(texts=[text])
    clusters = preds[0].get_clusters()          # list of clusters (each = list of mention strings)
    for i, c in enumerate(clusters, 1):
        print(f"Cluster {i}: {c}")
except Exception as e:
    USE_NEURAL = False
    print("fastcoref could not run, using rule-based fallback ->", e)

config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/393 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

[transformers] FCorefModel LOAD REPORT from: biu-nlp/f-coref
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fastcoref could not run, using rule-based fallback -> 'FCorefModel' object has no attribute 'all_tied_weights_keys'


In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")

def simple_resolve(text):
    doc = nlp(text)
    last_person, last_thing, out = None, None, []
    for tok in doc:
        word = tok.text
        if tok.ent_type_ == "PERSON":
            last_person = tok.text
        elif tok.pos_ == "NOUN" and tok.dep_ in ("dobj", "nsubj"):
            last_thing = tok.text
        elif tok.lower_ in ("he", "she") and last_person:
            word = last_person
        elif tok.lower_ == "it" and last_thing:
            word = "the " + last_thing
        out.append(word + tok.whitespace_)
    return "".join(out)

print("Original :", text)
print("Resolved :", simple_resolve(text))

Original : Rahul bought a laptop yesterday. He said it was very fast. Priya liked the laptop too, so she bought it for her sister.
Resolved : Rahul bought a laptop yesterday. He said the laptop was very fast. Priya liked the laptop too, so she bought the laptop for her sister.
